In [ ]:
# ============================================================
# CELL 1: Install Libraries
# ============================================================
#@title # 🎵 AI Music Generation with LSTM
#@markdown This notebook trains an LSTM on Bach's Chorales to generate new music.
#@markdown Everything runs in the cloud - no local GPU needed!

# Install system packages for audio synthesis
!apt-get update -qq && apt-get install -y -qq fluidsynth
!pip install -q music21 pretty_midi midi2audio

# For reproducibility and potential magenta use
!pip install -q magenta

# Standard imports
import tensorflow as tf
import numpy as np
import pandas as pd
import os
import sys
import random
import pickle
import time
import warnings
warnings.filterwarnings('ignore')

from tqdm import tqdm
import matplotlib.pyplot as plt
from collections import Counter
from itertools import chain
import copy

print("✅ All libraries installed successfully!")

In [ ]:
# ============================================================
# CELL 2: Configuration & Setup
# ============================================================
#@title ⚙️ Configuration & GPU Check

# --- QUICK TEST MODE ---
# Set to True to run a fast pipeline test (10% data, 5 epochs)
QUICK_TEST = False
# ---

# Mount Google Drive for saving models and outputs
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# Create directories
DRIVE_BASE = '/content/drive/MyDrive/music_generation'
MODEL_DIR = os.path.join(DRIVE_BASE, 'models')
OUTPUT_DIR = os.path.join(DRIVE_BASE, 'outputs')
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Set random seeds for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# GPU Check
print("\n--- GPU Information ---")
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"✅ GPU Available: {gpus}")
    print(f"   Using: {gpus[0]}")
    # Enable memory growth to avoid taking all VRAM
    try:
        tf.config.experimental.set_memory_growth(gpus[0], True)
    except RuntimeError as e:
        print(e)
else:
    print("⚠️ WARNING: No GPU found! Training will be extremely slow.")
    print("   Go to Runtime > Change runtime type > select 'T4 GPU'")

# Constants
SEQUENCE_LENGTH = 50
DEFAULT_DURATION = 0.5  # quarter note
TEMPERATURE = 0.8

print(f"\nConfiguration:")
print(f"  QUICK_TEST: {QUICK_TEST}")
print(f"  SEQUENCE_LENGTH: {SEQUENCE_LENGTH}")
print(f"  Drive Base: {DRIVE_BASE}")

In [ ]:
# ============================================================
# CELL 3: Data Loading — Bach Chorales (music21 Corpus)
# ============================================================
#@title 🎼 Section 1: Download & Load MIDI Data

from pathlib import Path

def load_bach_chorales(limit=None):
    """
    Loads MIDI files from the music21 corpus.
    Bach chorales are built-in and require no download.
    Returns a list of parsed music21 streams.
    """
    from music21 import corpus

    print("📂 Accessing music21 Corpus...")

    try:
        bach_paths = corpus.getComposer('bach')
    except Exception as e:
        print(f"⚠️ Could not get Bach paths: {str(e)}")
        bach_paths = []

    # Handle case where getComposer returns PosixPath objects
    if bach_paths and isinstance(bach_paths[0], Path):
        bach_paths = [str(p) for p in bach_paths]
    elif not bach_paths:
        print("   Trying 'corelli' as fallback...")
        try:
            bach_paths = corpus.getComposer('corelli')
            if bach_paths and isinstance(bach_paths[0], Path):
                bach_paths = [str(p) for p in bach_paths]
        except:
            pass

    if not bach_paths:
        # Last resort: try searching for chorales by name
        print("   Trying to get chorales by search...")
        try:
            bach_paths = corpus.search('bach', 'composer')
            if bach_paths:
                bach_paths = [str(p) for p in bach_paths]
        except:
            pass

    if not bach_paths:
        # Final fallback: use local corpus paths
        print("   Trying local corpus paths...")
        try:
            from music21 import common
            corpus_path = common.getMetadataBundleFile()
            # Manual path construction
            all_paths = list(corpus.getComposer('bach'))
            if all_paths:
                bach_paths = [str(p) for p in all_paths]
        except:
            print("   ❌ Could not find any corpus files")
            return []

    composer_name = "unknown"
    if bach_paths:
        # Extract composer name from first path
        try:
            path_parts = bach_paths[0].replace('\\', '/').split('/')
            for part in path_parts:
                if 'bach' in part.lower():
                    composer_name = 'bach'
                    break
                elif 'corelli' in part.lower():
                    composer_name = 'corelli'
                    break
            if composer_name == 'unknown' and 'bach' in str(bach_paths[0]).lower():
                composer_name = 'bach'
        except:
            pass

    print(f"   Found {len(bach_paths)} pieces by {composer_name}")

    if limit:
        bach_paths = bach_paths[:limit]
        print(f"   Using {limit} pieces for training.")

    successful_parses = []
    for path in tqdm(bach_paths, desc="Parsing MIDI files"):
        try:
            piece = corpus.parse(path)
            # Filter out non-MIDI elements (like pure text expressions)
            if piece.parts:
                successful_parses.append(piece)
            else:
                print(f"   ⚠️ Skipping {os.path.basename(str(path))}: No parts found")
        except Exception as e:
            print(f"   ⚠️ Skipping {os.path.basename(str(path))}: {str(e)[:100]}")

    print(f"✅ Successfully loaded {len(successful_parses)} pieces.")
    return successful_parses

# Determine number of pieces based on QUICK_TEST
N_PIECES = 10 if QUICK_TEST else None  # None = all
chorales = load_bach_chorales(limit=N_PIECES)

if not chorales:
    raise RuntimeError("No MIDI files loaded. Check music21 corpus availability.")

In [ ]:
# ============================================================
# CELL 4: MIDI Preprocessing — Notes & Chords Extraction
# ============================================================
#@title 🎹 Section 2: Extract Notes & Chords

from music21 import note, chord, stream, instrument

def parse_midi(score):
    """
    Extracts a flat list of note and chord events from a music21 stream.
    Handles multiple parts and instruments.
    """
    events = []
    try:
        # Ensure we have parts to iterate over
        parts = score.parts if score.parts else [score]

        for part in parts:
            # Flatten all measures and notes
            elements = part.flatten().notesAndRests
            for element in elements:
                if isinstance(element, note.Note):
                    # Store pitch with octave
                    events.append(str(element.pitch))
                elif isinstance(element, chord.Chord):
                    # Store chord notes joined by '.'
                    chord_notes = '.'.join(sorted([str(p) for p in element.pitches]))
                    events.append(chord_notes)
                elif isinstance(element, note.Rest):
                    # Represent rests as a special token
                    events.append('REST')
    except Exception as e:
        print(f"   ⚠️ Error parsing element: {str(e)[:80]}")
        return []

    return events

# Process all chorales
all_notes = []
print("🎶 Extracting notes from all pieces...")
for piece in tqdm(chorales, desc="Processing Scores"):
    try:
        notes = parse_midi(piece)
        all_notes.extend(notes)
    except Exception as e:
        print(f"   ⚠️ Failed to process piece: {str(e)[:100]}")

print(f"\n✅ Total note events extracted: {len(all_notes)}")

# Display statistics
unique_notes = sorted(set(all_notes))
print(f"   Unique note symbols: {len(unique_notes)}")
print(f"   Sample notes (first 20): {all_notes[:20]}")
print(f"   Most common notes: {Counter(all_notes).most_common(10)}")

if len(unique_notes) < 5:
    raise RuntimeError("Too few unique notes extracted. Check MIDI parsing logic.")

In [ ]:
# ============================================================
# CELL 5: Sequence Preparation & Encoding
# ============================================================
#@title 📊 Section 3: Create Training Sequences

def prepare_sequences(notes, sequence_length=SEQUENCE_LENGTH):
    """
    Converts a flat list of note strings into encoded input/output pairs
    for LSTM training.
    """
    print("🔢 Creating vocabulary mapping...")
    # Get sorted unique notes for deterministic mapping
    unique_notes = sorted(set(notes))
    vocab_size = len(unique_notes)

    # Create mapping dictionaries
    note_to_int = {note: i for i, note in enumerate(unique_notes)}
    int_to_note = {i: note for i, note in enumerate(unique_notes)}

    # Encode all notes to integers
    encoded_notes = [note_to_int[n] for n in notes]

    # Create sequences
    network_input = []
    network_output = []

    print(f"📏 Creating sequences of length {sequence_length}...")
    for i in tqdm(range(len(encoded_notes) - sequence_length), desc="Generating Sequences"):
        seq_in = encoded_notes[i:i + sequence_length]
        seq_out = encoded_notes[i + sequence_length]
        network_input.append(seq_in)
        network_output.append(seq_out)

    # Convert to numpy arrays
    X = np.array(network_input)
    y = network_output

    # One-hot encode the output
    y_onehot = tf.keras.utils.to_categorical(y, num_classes=vocab_size)

    print(f"\n✅ Sequence preparation complete:")
    print(f"   Input shape: {X.shape}")
    print(f"   Output shape: {y_onehot.shape}")
    print(f"   Vocabulary size: {vocab_size}")
    print(f"   Sample input sequence (encoded): {X[0][:10]}...")
    print(f"   Sample output (one-hot index): {np.argmax(y_onehot[0])} -> '{int_to_note[np.argmax(y_onehot[0])]}'")

    return X, y_onehot, note_to_int, int_to_note, vocab_size

# Prepare data
X, y, note_to_int, int_to_note, vocab_size = prepare_sequences(all_notes)

# Optional: Reduce data for quick testing
if QUICK_TEST:
    test_size = min(int(0.1 * len(X)), 5000)
    print(f"\n⚡ QUICK TEST MODE: Using {test_size} sequences out of {len(X)}")
    indices = np.random.choice(len(X), test_size, replace=False)
    X = X[indices]
    y = y[indices]

In [ ]:
# ============================================================
# CELL 6: Train/Validation Split & Data Normalization
# ============================================================
#@title 📚 Section 4: Data Preparation

from sklearn.model_selection import train_test_split

# Shuffle before splitting (important for good validation)
indices = np.random.permutation(len(X))
X = X[indices]
y = y[indices]

# Split data
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=SEED
)

print("📚 Data Split:")
print(f"   Training samples: {X_train.shape[0]}")
print(f"   Validation samples: {X_val.shape[0]}")

# Reshape input for LSTM: [samples, time_steps, features]
X_train = X_train.reshape((X_train.shape[0], X_train.shape[1], 1))
X_val = X_val.reshape((X_val.shape[0], X_val.shape[1], 1))

print(f"\n🔄 Reshaped for LSTM:")
print(f"   X_train shape: {X_train.shape}")
print(f"   X_val shape: {X_val.shape}")

# Normalize input to [0, 1] range
X_train = X_train.astype(np.float32) / float(vocab_size)
X_val = X_val.astype(np.float32) / float(vocab_size)

print(f"   Input normalized to [0, 1]")
print(f"   X_train range: [{X_train.min():.4f}, {X_train.max():.4f}]")

In [ ]:
# ============================================================
# CELL 7: Build LSTM Model (PROVEN ARCHITECTURE)
# ============================================================
#@title 🤖 Section 5: Model Architecture

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, BatchNormalization, Input
from tensorflow.keras.optimizers import Adam

def build_model(sequence_length, vocab_size, learning_rate=0.001):
    """
    LSTM network that reached 57% accuracy in first training run.
    """
    model = Sequential([
        Input(shape=(sequence_length, 1)),
        LSTM(256, return_sequences=True),
        Dropout(0.3),
        LSTM(256, return_sequences=False),
        Dropout(0.3),
        Dense(256, activation='relu'),
        BatchNormalization(),
        Dense(vocab_size, activation='softmax')
    ])

    optimizer = Adam(learning_rate=learning_rate)
    model.compile(
        loss='categorical_crossentropy',
        optimizer=optimizer,
        metrics=['accuracy']
    )
    return model

print("🏗️ Building model (proven architecture)...")
model = build_model(SEQUENCE_LENGTH, vocab_size)
model.summary()
print(f"\n✅ Model built with {model.count_params():,} parameters")
print(f"   This architecture reached 57% accuracy previously")

In [ ]:
# ============================================================
# CELL 8: Training Setup — 100 Epochs with Regular Saves
# ============================================================
#@title ⚡ Section 6: Training Configuration

from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, ReduceLROnPlateau, CSVLogger
)

# ============================================
# TRAINING PARAMETERS
# ============================================
if QUICK_TEST:
    EPOCHS = 5
    BATCH_SIZE = 64
    print("⚡ QUICK TEST MODE: 5 epochs, batch size 64")
else:
    EPOCHS = 100
    BATCH_SIZE = 64

print(f"\n📋 Training Configuration:")
print(f"   Max epochs: {EPOCHS}")
print(f"   Batch size: {BATCH_SIZE}")
print(f"   Training samples: {X_train.shape[0]:,}")
print(f"   Validation samples: {X_val.shape[0]:,}")
print(f"   Sequence length: {SEQUENCE_LENGTH}")
print(f"   Vocabulary size: {vocab_size}")

# ============================================
# CALLBACKS — WITH SAVE EVERY 5 EPOCHS
# ============================================
checkpoint_path = os.path.join(MODEL_DIR, 'best_model.keras')
periodic_path = os.path.join(MODEL_DIR, 'checkpoint_epoch_{epoch:03d}.keras')

# Calculate how many batches = 5 epochs
save_freq_batches = 5 * (len(X_train) // BATCH_SIZE)

callbacks = [
    # 1. Save the BEST model (lowest validation loss)
    ModelCheckpoint(
        checkpoint_path,
        monitor='val_loss',
        save_best_only=True,
        mode='min',
        verbose=1
    ),

    # 2. Save checkpoint EVERY 5 EPOCHS (safety backup)
    ModelCheckpoint(
        periodic_path,
        save_freq=save_freq_batches,   # Save every 5 epochs worth of batches
        save_best_only=False,           # Save regardless of performance
        verbose=1
    ),

    # 3. Stop if no improvement for 20 epochs
    EarlyStopping(
        monitor='val_loss',
        patience=20,
        min_delta=0.01,
        restore_best_weights=True,
        verbose=1
    ),

    # 4. Reduce learning rate when plateau
    ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.5,
        patience=10,
        min_delta=0.01,
        min_lr=1e-6,
        verbose=1
    ),

    # 5. Log training metrics
    CSVLogger(os.path.join(MODEL_DIR, 'training_log.csv'))
]

print(f"\n📊 Callbacks configured:")
print(f"   ✅ Save BEST model: best_model.keras")
print(f"   ✅ Save EVERY 5 epochs: checkpoint_epoch_XXX.keras")
print(f"   ✅ Save frequency: every {save_freq_batches} batches (~5 epochs)")
print(f"   ✅ EarlyStopping: patience=20, min_delta=0.01")
print(f"   ✅ ReduceLROnPlateau: patience=10, min_delta=0.01")
print(f"\n💡 If laptop dies, checkpoints saved at epochs: 5, 10, 15, 20, 25...")

In [ ]:
# ============================================================
# CELL 9: Train the Model
# ============================================================
#@title 🚀 Section 7: Start Training

import time as time_module

print("="*60)
print("🔥 STARTING TRAINING — 100 EPOCHS")
print("="*60)
print(f"   Training samples: {X_train.shape[0]:,}")
print(f"   Validation samples: {X_val.shape[0]:,}")
print(f"   GPU: {tf.config.list_physical_devices('GPU')}")
print(f"   Checkpoints saved every 5 epochs")
print(f"   Best model saved on improvement")
print("="*60)
print()

start_time = time_module.time()

try:
    history = model.fit(
        X_train, y_train,
        batch_size=BATCH_SIZE,
        epochs=EPOCHS,
        validation_data=(X_val, y_val),
        callbacks=callbacks,
        verbose=1
    )

    training_time = time_module.time() - start_time
    actual_epochs = len(history.history['loss'])

    print(f"\n{'='*60}")
    print(f"✅ TRAINING COMPLETE!")
    print(f"{'='*60}")
    print(f"   ⏱️  Time: {training_time/60:.1f} minutes")
    print(f"   📊 Epochs completed: {actual_epochs}/{EPOCHS}")
    print(f"   📉 Best training loss: {min(history.history['loss']):.4f}")
    print(f"   📈 Best validation loss: {min(history.history['val_loss']):.4f}")
    print(f"   📊 Best training accuracy: {max(history.history['accuracy']):.4f}")
    print(f"   🎯 Best validation accuracy: {max(history.history['val_accuracy']):.4f}")

    # Quality rating
    val_acc = max(history.history['val_accuracy'])
    if val_acc >= 0.70:
        quality = "🌟 EXCELLENT"
    elif val_acc >= 0.60:
        quality = "✅ GOOD"
    elif val_acc >= 0.50:
        quality = "👍 DECENT"
    else:
        quality = "⚠️ NEEDS IMPROVEMENT"
    print(f"   📊 Quality: {quality}")

    # Save final model
    final_path = os.path.join(MODEL_DIR, 'final_model.keras')
    model.save(final_path)
    print(f"\n💾 Models saved to Drive:")
    print(f"   Best: {os.path.join(MODEL_DIR, 'best_model.keras')}")
    print(f"   Final: {final_path}")
    print(f"   Periodic checkpoints: checkpoint_epoch_*.keras")

except KeyboardInterrupt:
    print(f"\n⚠️  Training interrupted by user!")
    emergency_path = os.path.join(MODEL_DIR, 'emergency_save.keras')
    model.save(emergency_path)
    print(f"   Emergency save: {emergency_path}")
    history = None

except Exception as e:
    print(f"\n❌ Training error: {str(e)}")
    emergency_path = os.path.join(MODEL_DIR, 'crash_save.keras')
    try:
        model.save(emergency_path)
        print(f"   Crash save: {emergency_path}")
    except:
        pass
    raise e

# ============================================
# RESUME INSTRUCTIONS (if needed)
# ============================================
print(f"\n{'='*60}")
print(f"📋 IF TRAINING IS INTERRUPTED:")
print(f"{'='*60}")
print(f"Find your latest checkpoint in:")
print(f"   {MODEL_DIR}")
print(f"Files: 'checkpoint_epoch_XXX.keras' (every 5 epochs)")
print(f"Best:  'best_model.keras'")
print(f"")
print(f"To resume, run:")
print(f"   model = load_model('checkpoint_epoch_XXX.keras')")
print(f"   Then run Cell 9 again")
print(f"{'='*60}")

In [ ]:
# ============================================================
# CELL 10: Plot Training History
# ============================================================
#@title 📈 Section 8: Training Visualization

def plot_training_history(history):
    """
    Plots training and validation loss/accuracy curves.
    """
    if history is None or len(history.history) == 0:
        print("No training history to plot.")
        return

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))

    # Loss plot
    axes[0].plot(history.history['loss'], label='Training Loss', linewidth=2)
    axes[0].plot(history.history['val_loss'], label='Validation Loss', linewidth=2)
    axes[0].set_title('Model Loss', fontsize=14)
    axes[0].set_ylabel('Loss')
    axes[0].set_xlabel('Epoch')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    # Accuracy plot
    axes[1].plot(history.history['accuracy'], label='Training Accuracy', linewidth=2)
    axes[1].plot(history.history['val_accuracy'], label='Validation Accuracy', linewidth=2)
    axes[1].set_title('Model Accuracy', fontsize=14)
    axes[1].set_ylabel('Accuracy')
    axes[1].set_xlabel('Epoch')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'training_curves.png'), dpi=100, bbox_inches='tight')
    plt.show()

    # Print final metrics
    final_train_loss = history.history['loss'][-1]
    final_val_loss = history.history['val_loss'][-1]
    print(f"\n📊 Final Metrics:")
    print(f"   Training Loss: {final_train_loss:.4f}")
    print(f"   Validation Loss: {final_val_loss:.4f}")

plot_training_history(history)

In [ ]:
# ============================================================
# CELL: CORRECTED Music Generation (Fix for Single Note Bug)
# ============================================================
#@title 🎹 Generate Music (FIXED VERSION)

import numpy as np
import random
from tqdm import tqdm

print("🎵 Starting CORRECTED music generation...")

# Reload model
from tensorflow.keras.models import load_model
model = load_model('/content/drive/MyDrive/music_generation/models/best_model.keras')
print("✅ Model loaded")

# IMPORTANT FIX: Use the raw encoded sequences, not the normalized ones
# The bug was that the seed was being normalized twice or incorrectly

def generate_music_fixed(model, note_to_int, int_to_note, vocab_size,
                         sequence_length=50, num_notes=300, temperature=1.2):
    """
    FIXED generation with proper temperature sampling and seed handling.
    """

    # Create a random seed from the vocabulary directly
    # This is the key fix - use integer indices, not normalized values
    seed_indices = np.random.randint(0, vocab_size, sequence_length)

    # Reshape for model input: (1, sequence_length, 1)
    pattern = seed_indices.reshape(1, sequence_length, 1)

    # Normalize ONLY for model input (values should be 0-1 range)
    pattern_normalized = pattern.astype(np.float32) / float(vocab_size)

    prediction_output = []

    print(f"Generating {num_notes} notes with temperature={temperature}...")

    for note_idx in tqdm(range(num_notes), desc="Composing"):
        # Get prediction from normalized input
        prediction = model.predict(pattern_normalized, verbose=0)[0]

        # Apply temperature sampling
        # Clip to avoid log(0)
        prediction = np.clip(prediction, 1e-7, 1 - 1e-7)
        logits = np.log(prediction) / temperature
        exp_logits = np.exp(logits)
        probabilities = exp_logits / np.sum(exp_logits)

        # Sample from the distribution
        next_index = np.random.choice(vocab_size, p=probabilities)

        # Decode to note string
        next_note = int_to_note[next_index]
        prediction_output.append(next_note)

        # Update pattern with the new note (keep as integers)
        pattern = np.append(pattern[:, 1:, :], [[[next_index]]], axis=1)

        # Re-normalize for next prediction
        pattern_normalized = pattern.astype(np.float32) / float(vocab_size)

    return prediction_output

# Generate with MULTIPLE temperatures and pick the best one
temperatures = [0.8, 1.0, 1.2, 1.5]
all_generations = {}

for temp in temperatures:
    print(f"\n{'='*50}")
    print(f"Testing temperature: {temp}")
    print(f"{'='*50}")

    notes = generate_music_fixed(
        model=model,
        note_to_int=note_to_int,
        int_to_note=int_to_note,
        vocab_size=vocab_size,
        sequence_length=SEQUENCE_LENGTH,
        num_notes=300,
        temperature=temp
    )

    # Analyze quality
    unique_count = len(set(notes))
    rest_count = notes.count('REST')
    chord_count = sum(1 for n in notes if '.' in n)

    print(f"\n📊 Stats for temp={temp}:")
    print(f"   Total notes: {len(notes)}")
    print(f"   Unique notes: {unique_count}")
    print(f"   Rest count: {rest_count}")
    print(f"   Chord count: {chord_count}")
    print(f"   Sample: {' → '.join(notes[:15])}...")

    all_generations[temp] = notes

# Pick the best generation (most variety, not too many rests)
def score_generation(notes):
    unique = len(set(notes))
    rest_ratio = notes.count('REST') / len(notes)
    # Penalize too many rests, reward variety
    return unique * (1 - rest_ratio)

best_temp = max(temperatures, key=lambda t: score_generation(all_generations[t]))
generated_notes = all_generations[best_temp]

print(f"\n{'='*50}")
print(f"✅ BEST GENERATION: temperature={best_temp}")
print(f"   Unique notes: {len(set(generated_notes))}")
print(f"   Sample: {' → '.join(generated_notes[:20])}...")
print(f"{'='*50}")

In [ ]:
# ============================================================
# CELL: Create MIDI from Generated Notes (FIXED)
# ============================================================
#@title 🎼 Create MIDI File (FIXED VERSION)

from music21 import stream, note, chord, tempo, meter, instrument
import random

def create_midi_fixed(prediction_output, output_file="generated_music_fixed.mid"):
    """
    FIXED MIDI creation with proper error handling and note validation.
    """
    print(f"🎼 Creating MIDI from {len(prediction_output)} notes...")

    # Create score
    generated_stream = stream.Score()
    generated_stream.insert(0, tempo.MetronomeMark(number=100))
    generated_stream.insert(0, meter.TimeSignature('4/4'))

    # Create piano part
    part = stream.Part()
    part.insert(0, instrument.Piano())

    current_offset = 0.0
    notes_added = 0

    for i, event in enumerate(prediction_output):
        try:
            # Vary duration for naturalness
            duration = random.choice([0.25, 0.5, 0.5, 0.5, 0.75, 1.0])

            if event == 'REST':
                # Add a rest
                r = note.Rest()
                r.duration.quarterLength = duration
                part.insert(current_offset, r)
                notes_added += 1

            elif '.' in event and len(event) > 1:
                # It's a chord
                chord_notes = []
                for pitch_str in event.split('.'):
                    pitch_str = pitch_str.strip()
                    if pitch_str and pitch_str != 'REST':
                        try:
                            n = note.Note(pitch_str)
                            chord_notes.append(n)
                        except:
                            continue

                if len(chord_notes) > 0:
                    c = chord.Chord(chord_notes)
                    c.duration.quarterLength = duration
                    part.insert(current_offset, c)
                    notes_added += 1
                else:
                    # Fallback to rest
                    part.insert(current_offset, note.Rest())

            elif event and event != 'REST':
                # Single note
                try:
                    n = note.Note(event)
                    n.duration.quarterLength = duration
                    part.insert(current_offset, n)
                    notes_added += 1
                except:
                    # Invalid note, insert rest
                    part.insert(current_offset, note.Rest())
            else:
                # Empty or invalid, insert rest
                part.insert(current_offset, note.Rest())

        except Exception as e:
            # On error, insert a rest to maintain timing
            try:
                part.insert(current_offset, note.Rest())
            except:
                pass

        current_offset += duration

    generated_stream.insert(0, part)

    # Write MIDI
    try:
        generated_stream.write('midi', fp=output_file)
        file_size = os.path.getsize(output_file)
        print(f"✅ MIDI saved: {output_file}")
        print(f"   File size: {file_size} bytes")
        print(f"   Notes added: {notes_added}")

        # Verify by reading back
        verify = converter.parse(output_file)
        verify_notes = len(verify.flatten().notes)
        print(f"   Verified notes in file: {verify_notes}")

    except Exception as e:
        print(f"❌ Error writing MIDI: {e}")
        # Try alternative method
        try:
            mf = generated_stream.write('midi')
            with open(output_file, 'wb') as f:
                f.write(mf)
            print(f"✅ MIDI saved (alternative method): {output_file}")
        except Exception as e2:
            print(f"❌ Complete failure: {e2}")
            return None

    return output_file

# Import for verification
from music21 import converter

# Save MIDI
output_path = '/content/drive/MyDrive/music_generation/outputs/generated_music_fixed.mid'
midi_file = create_midi_fixed(generated_notes, output_path)

# Also save locally
local_midi = '/content/generated_music_fixed.mid'
create_midi_fixed(generated_notes, local_midi)

print(f"\n✅ MIDI files created!")
print(f"   Google Drive: {output_path}")
print(f"   Local: {local_midi}")

In [ ]:
# ============================================================
# CELL: Convert to Audio & Play (FIXED)
# ============================================================
#@title 🔊 Play Generated Music

from midi2audio import FluidSynth
from IPython.display import Audio, display

# Convert MIDI to WAV
wav_path = '/content/generated_music_fixed.wav'
print("🔊 Converting MIDI to audio...")

try:
    fs = FluidSynth()
    fs.midi_to_audio(local_midi, wav_path)
    print(f"✅ WAV created: {wav_path}")

    # Play in notebook
    print("\n🎧 Your generated music:")
    display(Audio(wav_path))

    # Save to Drive
    drive_wav = '/content/drive/MyDrive/music_generation/outputs/generated_music_fixed.wav'
    import shutil
    shutil.copy(wav_path, drive_wav)
    print(f"✅ Saved to Drive: {drive_wav}")

except Exception as e:
    print(f"⚠️ Audio conversion failed: {e}")
    print("But MIDI is ready! Download and play with VLC.")

In [ ]:
# ============================================================
# CELL 14: Results Summary
# ============================================================
#@title 📊 Results Summary

print("="*60)
print("🎵 AI MUSIC GENERATION RESULTS")
print("="*60)

print(f"\n📊 Data Statistics:")
print(f"   Total pieces processed: {len(chorales)}")
print(f"   Total note events: {len(all_notes)}")
print(f"   Unique note symbols: {vocab_size}")
print(f"   Training sequences: {X_train.shape[0]}")
print(f"   Validation sequences: {X_val.shape[0]}")

print(f"\n🤖 Model Architecture:")
print(f"   Input shape: {model.input_shape}")
print(f"   Output shape: {model.output_shape}")
print(f"   Total parameters: {model.count_params():,}")

print(f"\n📈 Training Results:")
if history and hasattr(history, 'history'):
    final_train_loss = history.history['loss'][-1]
    final_val_loss = history.history['val_loss'][-1]
    final_train_acc = history.history['accuracy'][-1]
    final_val_acc = history.history['val_accuracy'][-1]
    print(f"   Final Training Loss: {final_train_loss:.4f}")
    print(f"   Final Validation Loss: {final_val_loss:.4f}")
    print(f"   Final Training Accuracy: {final_train_acc:.4f}")
    print(f"   Final Validation Accuracy: {final_val_acc:.4f}")
    print(f"   Training time: {training_time/60:.1f} minutes")

print(f"\n🎶 Generated Music:")
print(f"   Notes generated: {len(generated_notes)}")
print(f"   Temperature used: {TEMPERATURE}")

print(f"\n📁 Output Files:")
print(f"   MIDI: {midi_file}")
if 'wav_file' in locals() and wav_file and os.path.exists(wav_file):
    print(f"   WAV: {wav_file}")
print(f"   Model checkpoint: {os.path.join(MODEL_DIR, 'best_model.keras')}")
print(f"   Google Drive: {DRIVE_BASE}")

print("\n" + "="*60)
print("✅ Pipeline completed successfully!")
print("="*60)

# ============================================================
# CELL 15: Troubleshooting Guide (Markdown)
# ============================================================
#@title 🛠️ Troubleshooting Guide (Click to Expand)

troubleshooting_text = """
## 🛠️ Common Issues & Solutions

### 1. **Runtime Disconnection / Training Too Slow**
- **Symptom:** Colab disconnects or training takes >2 hours
- **Solution:**
  - Enable `QUICK_TEST = True` at the top (Cell 2)
  - Reduce `EPOCHS` to 20 in Cell 8
  - Ensure you're using a T4 GPU (Runtime → Change runtime type)

### 2. **"No module named 'music21'"**
- **Solution:** Re-run Cell 1 (Install Libraries) and check for errors

### 3. **MIDI Generation is Monotone/Repeating**
- **Symptom:** Same note repeats endlessly
- **Solution:**
  - Increase temperature to 0.9 or 1.0 in Cell 2
  - Train for more epochs (up to 100)
  - Use more data (set `N_PIECES = None` in Cell 3)

### 4. **FluidSynth Audio Conversion Fails**
- **Symptom:** WAV file not created
- **Solution:**
  - The notebook includes a fallback method
  - MIDI file is still valid - download it and play locally
  - Use a free online MIDI-to-MP3 converter

### 5. **Memory Errors (OOM)**
- **Symptom:** "Resource exhausted" or kernel dies
- **Solution:**
  - Reduce `SEQUENCE_LENGTH` to 25
  - Reduce batch size to 32
  - Enable `QUICK_TEST` mode

### 6. **Bad Note Sequences Generated**
- **Symptom:** Unmusical output, lots of dissonance
- **Solution:**
  - Bach chorales are your training data - output will sound Baroque
  - For modern music, try the Magenta dataset (Method C in description)
  - More training epochs usually helps

### 7. **Google Drive Mount Issues**
- **Symptom:** Permission errors
- **Solution:**
  - Follow the authorization link that appears
  - Ensure you're logged into the correct Google account
  - Files are still saved locally in `/content/`

### 8. **music21 Corpus Empty**
- **Symptom:** No Bach pieces found
- **Solution:**
  - The notebook automatically tries Corelli as fallback
  - Try manual download: `!pip install music21 --upgrade`
  - Alternative: Use the Magenta dataset with `!wget` as shown in Cell 3 comments
"""

from IPython.display import Markdown
Markdown(troubleshooting_text)